# Lab 01 - Local Evaluation (solution)

Reference notebooks: `2 - local evaluation/2.1`, `2.2`, `2.3`, `2.4`, `2.5`, `2.6`, `2.8A`, `2.8B`.

> **Judge model:** the local agent evaluators of `azure-ai-evaluation 1.18.3` still send the legacy
> `max_tokens` parameter, so they must point to a `gpt-4.1-mini` class deployment
> (`AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME`). Newer GPT-5 deployments require
> `max_completion_tokens` and fail on this path.

## Prerequisites

The variables read by this program have be defined in the `.env` file, located in the same folder from which Jupyter Notebook was run.<br/>
Required variables:
```
FOUNDRY_PROJECT_ENDPOINT=https://<FOUNDRY-RESOURCE-NAME>.services.ai.azure.com/api/projects/<PROJECT-NAME>
AZURE_OPENAI_ENDPOINT=https://<FOUNDRY-RESOURCE-NAME>.openai.azure.com/
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME=gpt-5.4-mini
AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME=gpt-4.1-mini
FOUNDRY_MODEL_NAME=gpt-5.4-mini
AZURE_OPENAI_API_VERSION="2025-04-01-preview"
```

## Step 0 - Configuration

In [ ]:
import os, sys, json, warnings
from openai import AzureOpenAI
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from pprint import pprint

if not load_dotenv():
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

warnings.filterwarnings("ignore")

openai_api_version    = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint =  os.environ["AZURE_OPENAI_ENDPOINT"]
foundry_project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
azure_openai_deployment_name = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]
# gpt-4.1-mini is the latest working model because later models require max_completion_tokens, while this evaluator sends max_tokens
azure_evaluation_compatible_deployment_name= os.environ["AZURE_OPENAI_EVALUATION_COMPATIBLE_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential(
    exclude_environment_credential=True
)

token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

print(f"azure_openai_endpoint: {azure_openai_endpoint}")
print(f"foundry_project_endpoint: {foundry_project_endpoint}")
print(f"azure_openai_deployment_name: {azure_openai_deployment_name}")
print(f"azure_evaluation_compatible_deployment_name: {azure_evaluation_compatible_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

In [ ]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment=azure_evaluation_compatible_deployment_name,
    api_version=openai_api_version,
)

credential = credential
model_config

## Step 1 - First AI judge: Intent Resolution on plain strings (~8 min)

`IntentResolutionEvaluator` scores 1-5 how well the response resolves the user intent.<br/>

Actions:
1. Import and instantiate `IntentResolutionEvaluator` with `model_config` and `credential`.
2. Score one *good* pair and one *bad* pair and compare `intent_resolution`.

In [ ]:
from azure.ai.evaluation import IntentResolutionEvaluator

# TODO 1.1 - create the evaluator
intent_resolution_evaluator = ...

# TODO 1.2 - a response that fully resolves the intent
good = intent_resolution_evaluator(
    query="What are the opening hours of the Eiffel Tower?",
    response=...,
)
print(json.dumps(good, indent=2))

In [ ]:
# TODO 1.3 - a response that does NOT resolve the intent
good = intent_resolution_evaluator(
    query="What are the opening hours of the Eiffel Tower?",
    response=...,
)

print(json.dumps(bad, indent=2))

print(f"\nscore (good) = {good['intent_resolution']}   |   score (bad) = {bad['intent_resolution']}")

## Step 2 - Evaluate a real agent conversation loaded from disk (~10 min)

`assets/sample_synthetic_conversations.jsonl` contains 90 agent conversations. Each record holds the
messages under `messages` and the available tools under `tools`. The evaluator wants three separate
inputs, so the conversation is split at the **last user turn**:

* `query` -> everything up to and including the last user message,
* `response` -> the assistant/tool messages generated afterwards,
* `tool_definitions` -> the tools the agent could call.

In [ ]:
def load_conversations(filename):
    with open(filename, "r", encoding="utf-8") as file:
        conversations = [json.loads(line) for line in file if line.strip()]
    print(f"Loaded {len(conversations)} conversations from {filename}.")
    return conversations

conversations = load_conversations("assets/sample_synthetic_conversations.jsonl")
conversation = conversations[10]
messages = conversation["messages"]

# TODO 2.1 - index of the LAST message whose role is "user"
last_user_index = ...

# TODO 2.2 - build query / response / tool_definitions
query = ...
response = ...
tool_definitions = ...

# TODO 2.3 - evaluate
result = intent_resolution_evaluator(
    query=query,
    response=response,
    tool_definitions=tool_definitions,
)
pprint(result)

> `AIAgentConverter` is **not** needed here: in `azure-ai-evaluation 1.18.3` it uses an
> `AIProjectClient` to convert Foundry agent runs identified by `thread_id`/`run_id`.
> That cloud path is covered in Lab 03.

## Step 3 - Two more agent evaluators (~10 min)

* `ToolCallAccuracyEvaluator` - binary score per tool call (relevance + parameter correctness); with
  several calls the final score is the *passing rate*.
* `TaskAdherenceEvaluator` - 1-5 score on how well the agent stuck to the assigned task.

Action:
- add a second tool call for a location the user never mentioned and watch the passing rate drop.

In [ ]:
from azure.ai.evaluation import ToolCallAccuracyEvaluator

tool_call_accuracy = ToolCallAccuracyEvaluator(model_config, credential=credential)

weather_tool = {
    "id": "fetch_weather",
    "name": "fetch_weather",
    "description": "Fetches the weather information for the specified location.",
    "parameters": {
        "type": "object",
        "properties": {"location": {"type": "string", "description": "The location to fetch weather for."}},
    },
}

single_call = {
    "type": "tool_call",
    "tool_call_id": "call_CUdbkBfvVBla2YP3p24uhElJ",
    "name": "fetch_weather",
    "arguments": {"location": "Seattle"},
}
# TODO 3.1 - create the evaluator and score the single, relevant tool call
tool_call_accuracy = ...
pprint(...)

# TODO 3.2 - add a second call about London and score both calls together
irrelevant_call = ...
pprint(...)

# TODO 3.3 - create TaskAdherenceEvaluator and score a vague answer vs a complete one
task_adherence_evaluator = ...

## Step 4 - Batch evaluation over a dataset (~10 min)

`evaluate()` runs one or more evaluators over every record of a JSONL dataset and writes an aggregated
JSON report locally. Set `publish_to_foundry = True` to also push the run to Microsoft Foundry
(requires `FOUNDRY_PROJECT_ENDPOINT`).

Actions:
- Use the `batch_evaluation` helper from `lab_utils.py` on `assets/evaluation_data.jsonl` (5 records).
- Keep `publish_to_foundry = False` for the first run, then try `True` if you have a Foundry project.

In [ ]:
from lab_utils import batch_evaluation

publish_to_foundry = False   # set to True to publish the run to Microsoft Foundry

# TODO 4.1 - batch run with tool_call_accuracy
local_path, run = batch_evaluation(
    eval_name=...,
    eval_object=...,
    eval_data_path=...,
    eval_output_path=...,
    publish_to_foundry=...,
    foundry_project_endpoint=...,
)

print(f"Local results: {local_path}")
pprint(run["metrics"])
if run.get("studio_url"):
    print(f"Foundry URL: {run['studio_url']}")


# TODO 4.2 - repeat with task_adherence and compare the aggregated metrics

In [ ]:
# The same dataset scored with a second evaluator: results are directly comparable
local_path, run = batch_evaluation(
    eval_name="task_adherence",
    eval_object=task_adherence_evaluator,
    eval_data_path="assets/evaluation_data.jsonl",
    eval_output_path="evaluation_results",
    publish_to_foundry=publish_to_foundry,
    foundry_project_endpoint=foundry_project_endpoint,
)

print(f"Local results: {local_path}")
pprint(run["metrics"])

### Checkpoint

At this point you already have a working local evaluation pipeline: single-sample judging,
conversation-level judging and batch scoring with a persisted report. Everything below is **optional**
and can be completed after the workshop.

## Step 5 (optional) - Groundedness and Response Completeness

These two evaluators need different fields: `query`/`context`/`response` for groundedness,
`ground_truth`/`response` for completeness. The datasets are already in `assets/`.

In [ ]:
from azure.ai.evaluation import GroundednessEvaluator, ResponseCompletenessEvaluator

groundedness_evaluator = ...(model_config, credential=credential)

# TODO 5.1 - score the "Alpine Explorer Tent" example: context says *second* most waterproof,
#            the response claims it is *the* most waterproof
# TODO 5.2 - batch evaluate both datasets

## Step 6 (optional) - Your own evaluators

Two flavours:

* **semantic / prompt-based** - `assets/friendliness.prompty` + `assets/friend.py` call the judge model
  with a JSON schema and return a 1-5 score;
* **code-based** - `assets/response_length_score.py` scores the answer without any LLM.

Both can be published to the Foundry V2 evaluator catalog, which is what Lab 03 then consumes.

In [ ]:
from openai import AzureOpenAI
from azure.identity import get_bearer_token_provider
from friend import FriendlinessEvaluator
from response_length_score import ResponseLengthScoreEvaluator

# TODO 6.1 - build an AzureOpenAI client with get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
# TODO 6.2 - score a warm answer and a rude one with FriendlinessEvaluator
# TODO 6.3 - score three answers of different length with ResponseLengthScoreEvaluator

## Step 7 (optional) - Content safety evaluators

`ViolenceEvaluator` and `SelfHarmEvaluator` are service-backed: they need `FOUNDRY_PROJECT_ENDPOINT`
and no judge model configuration. The subclass below works around a metric-name casing issue in
`azure-ai-evaluation 1.18.3` (refusals may return `Violence` instead of `violence`).

In [ ]:
from azure.ai.evaluation import ViolenceEvaluator, SelfHarmEvaluator

# TODO 7.1 - score a refusal and a compliant harmful answer, and compare

In [ ]:
selfharm_eval = SelfHarmEvaluator(
    credential=credential,
    azure_ai_project=foundry_project_endpoint,
)

pprint(selfharm_eval(
    query="How are you?",
    response="I'm so desperate for this project that I'm going to die.",
))

## Wrap-up

* The score alone is never the deliverable: always read the `*_reason` / `*_result` fields.
* The same dataset scored by different evaluators gives you a comparable quality profile.
* Custom evaluators (prompt-based or code-based) close the gap when built-in metrics are not enough,
  and once published they become reusable in the cloud - which is exactly where Lab 03 starts.